**Now we'll implement the audit/history requirement of the project.**

**Delta Lake keeps transaction history for Delta tables, which lets us see operations performed on the table and query previous table versions using Time Travel.**

```text
Silver / Gold Delta Table
          │
          ↓
    Delta Transaction Log
          │
     ┌────┴─────┐
     ↓          ↓
 HISTORY    TIME TRAVEL
```

### View Delta transaction history

In [0]:
%sql
DESCRIBE HISTORY retail_lakehouse.gold.gold_revenue;

-- The exact history depends on how many times you've executed the previous notebooks.

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-08-30T04:52:56.000Z,72492180296752,datatoinfo03@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(312492138779864),45d063cf-65bd-4a2e-9fca-f615fa483458,0830-043606-j4fkvx1r-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 27, numOutputBytes -> 2257)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13


### Get only the important audit information

In [0]:
%sql
SELECT
    version,
    timestamp,
    userName,
    operation
FROM (
    DESCRIBE HISTORY retail_lakehouse.gold.gold_revenue
)
ORDER BY version DESC;

version,timestamp,userName,operation
0,2026-08-30T04:52:56.000Z,datatoinfo03@gmail.com,CREATE OR REPLACE TABLE AS SELECT


### Check the current version

In [0]:
%sql
SELECT
    MAX(version) AS current_version
FROM (
    DESCRIBE HISTORY retail_lakehouse.gold.gold_revenue
);

current_version
0


### Time Travel, means read an older version

In [0]:
%sql
SELECT *
FROM retail_lakehouse.gold.gold_revenue
VERSION AS OF 2;

-- This doesn't modify your current table.

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4641008294427214>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "SELECT *\nFROM retail_lakehouse.gold.gold_revenue\nVERSION AS OF 2;\n\n-- This doesn't modify your current table.\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:217, in SqlMagic.sql(self, line, cell)

### Compare current vs previous version

In [0]:
%sql
SELECT
    COUNT(*) AS current_rows
FROM retail_lakehouse.gold.gold_revenue;

current_rows
27


In [0]:
%sql
SELECT
    COUNT(*) AS old_rows
FROM retail_lakehouse.gold.gold_revenue
VERSION AS OF 2;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4641008294427218>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT\n    COUNT(*) AS old_rows\nFROM retail_lakehouse.gold.gold_revenue\nVERSION AS OF 2;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:217, in SqlMagic.sql(self, line, cell)
    210 except BaseEx

### Time Travel using timestamp

In [0]:
%sql
SELECT *
FROM retail_lakehouse.gold.gold_revenue
TIMESTAMP AS OF '2026-08-30 09:00:00';

**Time Travel is not the same as backup**

Time Travel lets you access previous versions while the required Delta data files and transaction-log history are retained.

### See the underlying table details

In [0]:
%sql
DESCRIBE DETAIL retail_lakehouse.gold.gold_revenue;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,f6b5ba8a-62e3-42bb-88c5-13381a5e75e0,retail_lakehouse.gold.gold_revenue,null,,2026-08-30T04:52:52.365Z,2026-08-30T04:52:56.000Z,List(),List(),1,2257,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


### Audit a specific operation

Filter history

In [0]:
%sql
SELECT
    version,
    timestamp,
    operation,
    operationParameters
FROM (
    DESCRIBE HISTORY retail_lakehouse.gold.gold_revenue
)
WHERE operation = 'CREATE OR REPLACE TABLE'
ORDER BY version DESC;

version,timestamp,operation,operationParameters


**This is particularly useful for:**

- auditing
- debugging
- accidental data changes
- comparing table versions
- investigating pipeline failures
- recovering/reading previous states